# Phase 1: DistilGPT-2 Decoder Smoke Tests

Zero-shot prompting on Jigsaw comments — **D1** (per-label yes/no) and **D2** (toxic fill-in-blank).

Set `QUICK_MODE = True` in the setup cell for faster iteration (`20+20` eval sample).

In [ ]:
%matplotlib inline

import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

# Set True for faster runs (20/20 eval sample, fewer timing repeats).
QUICK_MODE = True
THRESHOLD = 0.5

_cwd = Path.cwd()
_bootstrap = _cwd.parent if _cwd.name == "notebooks" else _cwd
sys.path.insert(0, str(_bootstrap / "src"))

from data import add_toxic_any, load_train_df, verify_data_files
from decoder import predict_multilabel_d1, predict_toxic_d2
from device import get_device, sync_if_cuda
from eval_sample import get_balanced_sample, get_smoke_texts
from paths import (
    DECODER_SMOKE_SCORES_PATH,
    DISTILGPT2_MODEL_ID,
    LOCAL_DISTILGPT2_DIR,
    PROJECT_ROOT,
    SEED,
    TOXIC_LABELS,
    eval_n_each,
)

np.random.seed(SEED)
torch.manual_seed(SEED)

device = get_device()
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device} ({'GPU' if device.type == 'cuda' else 'CPU'})")
print(f"QUICK_MODE: {QUICK_MODE} (eval n_each={eval_n_each(QUICK_MODE)})")
print(f"PyTorch: {torch.__version__}")

## 1. Load data + balanced eval sample

In [ ]:
verify_data_files()
train_df = add_toxic_any(load_train_df())
sample = get_balanced_sample(train_df, quick_mode=QUICK_MODE)
texts = sample["comment_text"].astype(str).tolist()

print(f"Eval sample: {len(sample)} rows ({sample['toxic'].sum()} toxic, {(sample['toxic']==0).sum()} non-toxic)")

### Install model dependencies (run once)

If needed: `%pip install -r ../requirements-project.txt`, then restart kernel.

In [ ]:
# Uncomment and run once if needed:
# %pip install -r ../requirements-project.txt

## 2. Load DistilGPT-2

Local copy in `models/distilgpt2/` if present; otherwise Hugging Face. Prefetch:
`python scripts/download_distilgpt2.py`

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_path = str(LOCAL_DISTILGPT2_DIR) if LOCAL_DISTILGPT2_DIR.exists() else DISTILGPT2_MODEL_ID
print(f"Loading from: {model_path}")

tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

## 3. Hand-picked smoke tests

In [ ]:
smoke_texts = get_smoke_texts()
smoke_d1 = predict_multilabel_d1(model, tokenizer, smoke_texts, device)
smoke_d2 = predict_toxic_d2(model, tokenizer, smoke_texts, device)

smoke_view = smoke_d1.merge(smoke_d2, on="comment_text")
cols = ["comment_text", "d2_p_toxic", "d2_pred_logprob", "d2_pred_generate", "d2_disagree"]
cols += [f"d1_p_yes_{l}" for l in TOXIC_LABELS]
display(smoke_view[cols].round(3))

## 4. D1 — per-label yes/no on balanced sample

6 prompts per comment. Slow on CPU — use `QUICK_MODE` while debugging.

In [ ]:
print(f"Running D1 on {len(texts)} comments x {len(TOXIC_LABELS)} labels ...")
d1_df = predict_multilabel_d1(model, tokenizer, texts, device)
d1_df.head(3)

## 5. D2 — toxic fill-in-blank on balanced sample

In [ ]:
print(f"Running D2 on {len(texts)} comments ...")
d2_df = predict_toxic_d2(model, tokenizer, texts, device)
d2_df.head(3)

## 6. Sanity check — toxic label (D1 vs D2)

In [ ]:
compare = sample[["comment_text"] + TOXIC_LABELS].copy()
compare = compare.merge(d1_df, on="comment_text").merge(d2_df, on="comment_text")

compare["d1_p_toxic"] = compare["d1_p_yes_toxic"]
compare["d1_pred_toxic"] = compare["d1_pred_logprob_toxic"]
compare["d2_pred_toxic"] = compare["d2_pred_logprob"]

for prefix, score_col, pred_col in [
    ("D1", "d1_p_toxic", "d1_pred_toxic"),
    ("D2", "d2_p_toxic", "d2_pred_toxic"),
]:
    agree = (compare["toxic"] == compare[pred_col]).mean()
    print(f"{prefix} agreement vs gold (threshold={THRESHOLD} on score): {agree:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_df = compare.assign(gold_label=compare["toxic"].map({0: "non-toxic", 1: "toxic"}))

for ax, score_col, title in [
    (axes[0], "d1_p_toxic", "D1 P(yes) for toxic"),
    (axes[1], "d2_p_toxic", "D2 P(toxic token)"),
]:
    sns.violinplot(data=plot_df, x="gold_label", y=score_col, ax=ax, palette=["seagreen", "indianred"])
    ax.axhline(THRESHOLD, color="crimson", linestyle="--", linewidth=1)
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 7. Multi-label agreement (D1, all 6 labels)

In [ ]:
agree_rows = []
for label in TOXIC_LABELS:
    pred_col = f"d1_pred_logprob_{label}"
    agree_rows.append({
        "label": label,
        "agreement": (compare[label] == compare[pred_col]).mean(),
        "disagree_logprob_vs_gen": compare[f"d1_disagree_{label}"].mean(),
    })
agree_df = pd.DataFrame(agree_rows)
display(agree_df.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=agree_df, x="label", y="agreement", ax=ax, color="steelblue")
ax.set_ylim(0, 1)
ax.set_title("D1 log-prob agreement with gold (per label)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Log-prob vs generation disagreements

In [ ]:
disagree_rows = []
for label in TOXIC_LABELS:
    disagree_rows.append({
        "label": label,
        "rate": compare[f"d1_disagree_{label}"].mean(),
    })
disagree_df = pd.DataFrame(disagree_rows)
print(f"D2 toxic disagree rate: {compare['d2_disagree'].mean():.1%}")
display(disagree_df.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=disagree_df, x="label", y="rate", ax=ax, color="coral")
ax.set_title("D1: log-prob pred != greedy generation")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 9. Inference micro-benchmark

In [ ]:
from decoder import build_d1_prompt, score_yes_no

bench_text = texts[0]
bench_prompt = build_d1_prompt(bench_text, "toxic")
bench_repeats = 50 if QUICK_MODE else 200

for _ in range(3):
    score_yes_no(model, tokenizer, bench_prompt, device)
sync_if_cuda(device)

t0 = time.perf_counter()
for _ in range(bench_repeats):
    score_yes_no(model, tokenizer, bench_prompt, device)
sync_if_cuda(device)
ms_per_prompt = 1000 * (time.perf_counter() - t0) / bench_repeats

print(f"Model: {DISTILGPT2_MODEL_ID}")
print(f"Parameters: {n_params:,}")
print(f"Device: {device}")
print(f"D1 single prompt (1 label): {ms_per_prompt:.2f} ms")
print(f"D1 all 6 labels (estimate): {ms_per_prompt * 6:.2f} ms/comment")

## 10. Save decoder smoke scores

In [ ]:
out = compare.copy()
out["quick_mode"] = QUICK_MODE
out["model_id"] = DISTILGPT2_MODEL_ID
out["device"] = str(device)
out["threshold"] = THRESHOLD

DECODER_SMOKE_SCORES_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(DECODER_SMOKE_SCORES_PATH, index=False)
print(f"Saved {len(out)} rows to {DECODER_SMOKE_SCORES_PATH}")

## Next steps

- Join `encoder_smoke_scores.parquet` (notebook 01) with `decoder_smoke_scores.parquet` for side-by-side plots
- Deferred: formal eval notebook with train/val/test split
- Phase 2: `run_benchmark.py` — D3 few-shot, prompt ablations, full metrics